# `interrupt()`：动态暂停、人工恢复与工程边界

> 适用版本：本项目锁定的 **LangGraph 1.1.2** 与 **langgraph-checkpoint 4.0.1**。全部示例只使用本地确定性代码和 `InMemorySaver`，不调用大模型、网络、数据库或付费 API。

`interrupt()` 让节点在业务逻辑内部动态暂停，把问题、审批上下文或待编辑内容交给图外的人；调用者取得中断事件后，可在稍后以相同 `thread_id` 和 `Command(resume=...)` 恢复。

本章参考课程页面的功能清单重新设计代码，不复刻其原文或依赖外部模型。课程的“常见使用模式”概览列出 7 类动态模式；页面正文当前展开到第 6 类，本章仍按概览语义把第 7 类输入校验完整实现。

[功能对照参考：尚硅谷 AI 课程笔记《8. 中断》](https://xbsheng.github.io/atguigu-note/langgraph/%E8%AF%BE%E4%BB%B6/04-LangGraph%E4%B8%AD%E6%96%AD%E4%B8%8E%E5%B7%A5%E5%85%B7%E4%B8%8E%E9%83%A8%E7%BD%B2)


In [1]:
from importlib.metadata import version
from pprint import pprint
from typing import Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

print("LangGraph version:", version("langgraph"))
print("Checkpoint version:", version("langgraph-checkpoint"))


LangGraph version: 1.1.2
Checkpoint version: 4.0.1


## 1. 课程页面案例盘点与本章边界

### 1.1 动态 `interrupt` / HITL 核心案例

| 编号 | 场景 | 本章原创实现 |
| --- | --- | --- |
| 1 | 基础 HITL：一次暂停、一次人工补充 | 给工单补充人工处理说明 |
| 2 | 多个并行中断：按 `Interrupt.id` 分发恢复值 | 并行收集显示名和服务等级 |
| 3 | 审批模式：人工决定后续路径 | 审批课程发布，动态路由发布/拒绝节点 |
| 4 | 审核与编辑：人类修改机器草稿 | 本地规则生成草稿，再由人工返回修订稿 |
| 5 | 工具执行审批：副作用前确认 | 审批本地 `safe_add` 模拟工具，再进入独立执行节点 |
| 6 | 单节点串行中断：一个节点内依次提问 | 依次收集姓名、年龄、角色 |
| 7 | 人类输入校验：非法值再次中断 | 评分不在 1～5 时返回错误并再次暂停 |

### 1.2 直接相关的机制专题

课程还单独分析基础中断、多个并行中断，以及“同一超步中部分任务完成、部分任务中断”的 checkpoint。本章不重复堆叠相同代码，而是在每个案例中输出暂停 `StateSnapshot`，并额外实现一个部分超步案例，直接查看 `tasks[*].result` 与 `tasks[*].interrupts`。

### 1.3 使用规范与反例

本章统一覆盖四条规范：

1. 不要用宽泛 `try/except` 包住节点内的 `interrupt()`；
2. 恢复前不要动态改变同一节点中 `interrupt()` 的调用顺序或数量；
3. 中断载荷使用可序列化的简单值、列表或字典；
4. `interrupt()` 前的外部副作用会因节点重跑而重复，必须幂等，或移动到中断后的独立节点。

### 1.4 静态断点与排除项

`interrupt_before` / `interrupt_after` 是调试用的静态断点，不是接收人工反馈的业务 HITL。本章末尾保留一个本地对照案例，但不把它计入上述 7 类动态模式。

页面后续的 LangGraph CLI、本地 Agent Server、LangSmith Studio、AgentChatUI、`.env` 和部署配置属于部署主题；模型调用、真实天气工具等外部集成也不是讲清 `interrupt()` 所必需。本章只保留“工具执行前审批”这一 HITL 结构，并用本地确定性函数模拟工具。


## 2. 先把运行机制讲清楚

节点第一次执行到 `interrupt(payload)` 时，`interrupt()` 不会像普通函数那样返回。它通过 LangGraph 内部的控制流异常把执行权交回运行时；运行时捕获该信号，将可恢复信息写入 checkpointer，然后让本次 `invoke()` **携带 `__interrupt__` 返回**，而不是让 Python 调用一直阻塞。

checkpointer 保存的是 LangGraph 的 **State、待执行任务、任务结果/错误和中断元数据**，不是 Python 调用栈，也不是“源码第几行”的指针。恢复时，运行时按相同 `thread_id` 找到暂停线程，重新调度包含 `interrupt()` 的节点；节点从函数开头执行，运行到相同次序的 `interrupt()` 后，`Command(resume=value)` 中的 `value` 才成为该次 `interrupt()` 的返回值。

```mermaid
sequenceDiagram
    participant Caller as 图外调用者
    participant Runtime as LangGraph Runtime
    participant Node as 含 interrupt 的节点
    participant Saver as InMemorySaver

    Caller->>Runtime: invoke(initial_state, same thread_id)
    Runtime->>Node: 从节点函数开头执行
    Node-->>Runtime: interrupt(payload) 触发内部控制流异常
    Runtime->>Saver: 保存 State、tasks、interrupt 元数据
    Runtime-->>Caller: 返回 {__interrupt__: [...]}
    Note over Caller: invoke 已返回，并未一直阻塞
    Caller->>Runtime: invoke(Command(resume=value), same thread_id)
    Runtime->>Saver: 定位同一暂停线程
    Runtime->>Node: 从函数开头重新执行
    Node->>Node: 到同一次 interrupt，取得 value
    Node-->>Runtime: 提交 State 更新
    Runtime-->>Caller: 返回最终 State
```

这也解释了最重要的工程边界：`interrupt()` 前面的代码会再次执行。若那里写数据库、发邮件、扣款或追加外部日志，就必须按至少一次执行语义设计幂等。


## 3. 案例 1：基础 HITL——补充工单说明

场景：自动流程已创建工单，但需要人工补充处理说明。首次调用只提出问题；恢复值是一个结构化字典，并原样成为 `interrupt()` 的返回值。


In [2]:
class BasicState(TypedDict, total=False):
    ticket_id: str
    human_note: dict[str, str]


basic_entry_log: list[str] = []
basic_return_log: list[dict[str, str]] = []


def collect_human_note(state: BasicState) -> dict[str, object]:
    # 位于 interrupt() 前，恢复时会再次执行。
    basic_entry_log.append(state["ticket_id"])
    note = interrupt(
        {
            "kind": "ticket_note",
            "ticket_id": state["ticket_id"],
            "question": "请补充本次工单的人工处理说明",
        }
    )
    basic_return_log.append(note)
    return {"human_note": note}


basic_builder = StateGraph(BasicState)
basic_builder.add_node("collect_human_note", collect_human_note)
basic_builder.add_edge(START, "collect_human_note")
basic_builder.add_edge("collect_human_note", END)
basic_saver = InMemorySaver()
basic_graph = basic_builder.compile(checkpointer=basic_saver)
basic_config = {
    "configurable": {"thread_id": "interrupt-basic-input-v1"}
}

basic_first = basic_graph.invoke(
    {"ticket_id": "TICKET-001"},
    basic_config,
    durability="sync",
)
basic_paused = basic_graph.get_state(basic_config)

print("首次 invoke 完整原始返回值：")
pprint(basic_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(basic_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(basic_paused, sort_dicts=False)

basic_resume_value = {
    "operator": "Lin",
    "note": "已复核日志，允许进入下一步",
}
basic_command = Command(resume=basic_resume_value)
print("\n恢复使用的完整 Command：")
pprint(basic_command, sort_dicts=False)

basic_final = basic_graph.invoke(
    basic_command,
    basic_config,
    durability="sync",
)
basic_final_snapshot = basic_graph.get_state(basic_config)
print("\n恢复后的完整最终 State：")
pprint(basic_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(basic_final_snapshot, sort_dicts=False)
print("\n完整节点入口日志：")
pprint(basic_entry_log, sort_dicts=False)
print("\n完整 interrupt 返回日志：")
pprint(basic_return_log, sort_dicts=False)

assert basic_final["human_note"] == basic_resume_value
assert basic_return_log == [basic_resume_value]
assert len(basic_entry_log) == 2
assert basic_final_snapshot.next == ()

print("\n友好摘要：一次暂停已恢复；节点入口 2 次，interrupt 真正返回 1 次。")


首次 invoke 完整原始返回值：
{'ticket_id': 'TICKET-001',
 '__interrupt__': [Interrupt(value={'kind': 'ticket_note',
                                    'ticket_id': 'TICKET-001',
                                    'question': '请补充本次工单的人工处理说明'},
                             id='daed1c19b7a2a591cc37f3098c0592f4')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'ticket_note',
                  'ticket_id': 'TICKET-001',
                  'question': '请补充本次工单的人工处理说明'},
           id='daed1c19b7a2a591cc37f3098c0592f4')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={'ticket_id': 'TICKET-001'}, next=('collect_human_note',), config={'configurable': {'thread_id': 'interrupt-basic-input-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1044-6bc6-8000-068ea7b6cae9'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-21T09:46:59.430079+00:00', parent_config={'configurable': {'thread_id': 'interrupt-basic-input-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1043-691a-bfff

### 结果解读

首次结果只有输入 State 与 `__interrupt__`，没有 `human_note`；暂停快照的 `tasks` 和 `interrupts` 保留了恢复所需的中断元数据。恢复后节点从开头重跑，所以入口日志有两项，但 `interrupt()` 后的返回日志只有一项。完整最终 State 中的 `human_note` 与 `Command.resume` 完全相同。


## 4. 案例 2：多个并行中断——按中断 ID 恢复

两个并行节点分别收集显示名和服务等级。并行中断的返回顺序不应被当成业务契约；调用方读取每个 `Interrupt.value`，再构造 `Interrupt.id -> 恢复值` 映射，一次性恢复当前待处理的中断集合。


In [3]:
class ParallelState(TypedDict, total=False):
    display_name: str
    service_level: int


parallel_entry_log: list[str] = []


def collect_display_name(_: ParallelState) -> dict[str, object]:
    parallel_entry_log.append("display_name")
    value = interrupt(
        {
            "kind": "profile_field",
            "field": "display_name",
            "question": "请输入显示名",
        }
    )
    return {"display_name": value}


def collect_service_level(_: ParallelState) -> dict[str, object]:
    parallel_entry_log.append("service_level")
    value = interrupt(
        {
            "kind": "profile_field",
            "field": "service_level",
            "question": "请选择服务等级（1～3）",
        }
    )
    return {"service_level": value}


parallel_builder = StateGraph(ParallelState)
parallel_builder.add_node("collect_display_name", collect_display_name)
parallel_builder.add_node("collect_service_level", collect_service_level)
parallel_builder.add_edge(START, "collect_display_name")
parallel_builder.add_edge(START, "collect_service_level")
parallel_builder.add_edge("collect_display_name", END)
parallel_builder.add_edge("collect_service_level", END)
parallel_saver = InMemorySaver()
parallel_graph = parallel_builder.compile(checkpointer=parallel_saver)
parallel_config = {
    "configurable": {"thread_id": "interrupt-parallel-input-v1"}
}

parallel_first = parallel_graph.invoke(
    {}, parallel_config, durability="sync"
)
parallel_paused = parallel_graph.get_state(parallel_config)
print("首次 invoke 完整原始返回值：")
pprint(parallel_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(parallel_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(parallel_paused, sort_dicts=False)

parallel_answers = {
    "display_name": "图学习者",
    "service_level": 2,
}
parallel_resume_map = {
    item.id: parallel_answers[item.value["field"]]
    for item in parallel_first["__interrupt__"]
}
parallel_command = Command(resume=parallel_resume_map)
print("\n完整 Interrupt.id -> 恢复值映射：")
pprint(parallel_resume_map, sort_dicts=False)
print("\n恢复使用的完整 Command：")
pprint(parallel_command, sort_dicts=False)

parallel_final = parallel_graph.invoke(
    parallel_command,
    parallel_config,
    durability="sync",
)
parallel_final_snapshot = parallel_graph.get_state(parallel_config)
print("\n恢复后的完整最终 State：")
pprint(parallel_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(parallel_final_snapshot, sort_dicts=False)
print("\n完整并行节点入口日志：")
pprint(parallel_entry_log, sort_dicts=False)

assert len(parallel_first["__interrupt__"]) == 2
assert parallel_final == parallel_answers
assert len(parallel_entry_log) == 4
assert parallel_final_snapshot.next == ()

print("\n友好摘要：两个并行中断已按各自 ID 一次性恢复。")


首次 invoke 完整原始返回值：
{'__interrupt__': [Interrupt(value={'kind': 'profile_field',
                                    'field': 'display_name',
                                    'question': '请输入显示名'},
                             id='fa03536d350b87f245bcf3988e08908e'),
                   Interrupt(value={'kind': 'profile_field',
                                    'field': 'service_level',
                                    'question': '请选择服务等级（1～3）'},
                             id='f15c0d44f5c7a1afb058d2f72f7df549')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'profile_field',
                  'field': 'display_name',
                  'question': '请输入显示名'},
           id='fa03536d350b87f245bcf3988e08908e'),
 Interrupt(value={'kind': 'profile_field',
                  'field': 'service_level',
                  'question': '请选择服务等级（1～3）'},
           id='f15c0d44f5c7a1afb058d2f72f7df549')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={}, next=('collect_display_name', 'collect_s

### 结果解读

暂停快照包含两个带中断的并行任务。恢复映射不依赖中断列表的先后位置，而是用每个 `Interrupt.id` 精确投递。两个节点都经历“首次暂停 + 恢复重跑”，所以入口日志总计 4 项；最终 State 同时合并两个不同字段，不需要 reducer。


## 5. 案例 3：审批模式——恢复值决定后续路径

场景：发布课程前由人工批准或拒绝。审批节点返回 `Command(update=..., goto=...)`，把人工决定同时写入 State 并选择后续节点。真正模拟副作用的发布动作位于独立下游节点。


In [4]:
class ApprovalState(TypedDict, total=False):
    operation_id: str
    request: str
    decision: dict[str, str]
    outcome: str


approval_entry_log: list[str] = []
approval_return_log: list[dict[str, str]] = []
publish_effect_log: list[str] = []


def request_approval(
    state: ApprovalState,
) -> Command[Literal["publish_course", "reject_course"]]:
    approval_entry_log.append(state["operation_id"])
    decision = interrupt(
        {
            "kind": "approval_request",
            "operation_id": state["operation_id"],
            "question": f"是否批准：{state['request']}？",
            "allowed_actions": ["approve", "reject"],
        }
    )
    approval_return_log.append(decision)
    target = (
        "publish_course"
        if decision["action"] == "approve"
        else "reject_course"
    )
    return Command(update={"decision": decision}, goto=target)


def publish_course(state: ApprovalState) -> dict[str, str]:
    # 仅记录本地确定性日志，模拟被隔离到独立节点的外部动作。
    publish_effect_log.append(state["operation_id"])
    return {"outcome": f"已批准：{state['request']}"}


def reject_course(state: ApprovalState) -> dict[str, str]:
    return {"outcome": f"已拒绝：{state['request']}"}


approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("request_approval", request_approval)
approval_builder.add_node("publish_course", publish_course)
approval_builder.add_node("reject_course", reject_course)
approval_builder.add_edge(START, "request_approval")
approval_builder.add_edge("publish_course", END)
approval_builder.add_edge("reject_course", END)
approval_saver = InMemorySaver()
approval_graph = approval_builder.compile(checkpointer=approval_saver)
approval_config = {
    "configurable": {"thread_id": "interrupt-approval-v1"}
}

approval_first = approval_graph.invoke(
    {
        "operation_id": "COURSE-001",
        "request": "发布 interrupt 教程",
    },
    approval_config,
    durability="sync",
)
approval_paused = approval_graph.get_state(approval_config)
print("首次 invoke 完整原始返回值：")
pprint(approval_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(approval_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(approval_paused, sort_dicts=False)

approval_resume_value = {
    "action": "approve",
    "reviewer": "教学用户",
    "comment": "本地示例已核对",
}
approval_command = Command(resume=approval_resume_value)
print("\n恢复使用的完整 Command：")
pprint(approval_command, sort_dicts=False)

approval_final = approval_graph.invoke(
    approval_command,
    approval_config,
    durability="sync",
)
approval_final_snapshot = approval_graph.get_state(approval_config)
print("\n恢复后的完整最终 State：")
pprint(approval_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(approval_final_snapshot, sort_dicts=False)
print("\n完整审批节点入口日志：")
pprint(approval_entry_log, sort_dicts=False)
print("\n完整 interrupt 返回日志：")
pprint(approval_return_log, sort_dicts=False)
print("\n完整发布动作日志：")
pprint(publish_effect_log, sort_dicts=False)

assert approval_final["decision"] == approval_resume_value
assert approval_final["outcome"].startswith("已批准")
assert len(approval_entry_log) == 2
assert publish_effect_log == ["COURSE-001"]

print("\n友好摘要：审批节点重跑 1 次，批准路径的独立动作节点执行 1 次。")


首次 invoke 完整原始返回值：
{'operation_id': 'COURSE-001',
 'request': '发布 interrupt 教程',
 '__interrupt__': [Interrupt(value={'kind': 'approval_request',
                                    'operation_id': 'COURSE-001',
                                    'question': '是否批准：发布 interrupt 教程？',
                                    'allowed_actions': ['approve', 'reject']},
                             id='14bde038fbd4da7cf5bac0cbcad06334')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'approval_request',
                  'operation_id': 'COURSE-001',
                  'question': '是否批准：发布 interrupt 教程？',
                  'allowed_actions': ['approve', 'reject']},
           id='14bde038fbd4da7cf5bac0cbcad06334')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={'operation_id': 'COURSE-001', 'request': '发布 interrupt 教程'}, next=('request_approval',), config={'configurable': {'thread_id': 'interrupt-approval-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-107d-6e9e-8000-2aecbc09879f'}}, metadat

### 结果解读

`approval_return_log[0]` 与最终 State 的 `decision` 都是完整 resume 字典，直接证明恢复值成为 `interrupt()` 返回值。审批节点入口有两项，而独立的发布节点只有一项：把副作用移出中断节点，可以避免它因为“恢复到同一 interrupt”而必然重复；但失败重试、replay、time travel 或进程崩溃仍可能让下游节点重跑，因此真实发布接口仍应支持幂等键。


## 6. 案例 4：审核与编辑——人工修订本地草稿

原课程示例依赖模型生成内容；这里改为本地规则生成草稿，保留真正需要学习的 HITL 结构：上游生成、人工查看并编辑、恢复后提交修订稿。


In [5]:
class ReviewState(TypedDict, total=False):
    topic: str
    draft: str
    review: dict[str, str]
    final_text: str


draft_entry_log: list[str] = []
review_entry_log: list[str] = []


def create_local_draft(state: ReviewState) -> dict[str, str]:
    draft_entry_log.append(state["topic"])
    return {
        "draft": (
            f"主题：{state['topic']}。"
            "草稿结论：中断允许工作流等待人工反馈。"
        )
    }


def review_and_edit(state: ReviewState) -> dict[str, object]:
    review_entry_log.append(state["topic"])
    review = interrupt(
        {
            "kind": "content_review",
            "instruction": "请审核并返回修订后的正文",
            "draft": state["draft"],
        }
    )
    return {
        "review": review,
        "final_text": review["edited_text"],
    }


review_builder = StateGraph(ReviewState)
review_builder.add_node("create_local_draft", create_local_draft)
review_builder.add_node("review_and_edit", review_and_edit)
review_builder.add_edge(START, "create_local_draft")
review_builder.add_edge("create_local_draft", "review_and_edit")
review_builder.add_edge("review_and_edit", END)
review_saver = InMemorySaver()
review_graph = review_builder.compile(checkpointer=review_saver)
review_config = {
    "configurable": {"thread_id": "interrupt-review-edit-v1"}
}

review_first = review_graph.invoke(
    {"topic": "LangGraph interrupt"},
    review_config,
    durability="sync",
)
review_paused = review_graph.get_state(review_config)
print("首次 invoke 完整原始返回值：")
pprint(review_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(review_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(review_paused, sort_dicts=False)

review_resume_value = {
    "reviewer": "课程编辑",
    "edited_text": (
        "中断会让 invoke 返回事件；稍后使用同一 thread_id 恢复。"
    ),
}
review_command = Command(resume=review_resume_value)
print("\n恢复使用的完整 Command：")
pprint(review_command, sort_dicts=False)

review_final = review_graph.invoke(
    review_command,
    review_config,
    durability="sync",
)
review_final_snapshot = review_graph.get_state(review_config)
print("\n恢复后的完整最终 State：")
pprint(review_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(review_final_snapshot, sort_dicts=False)
print("\n完整草稿节点入口日志：")
pprint(draft_entry_log, sort_dicts=False)
print("\n完整审核节点入口日志：")
pprint(review_entry_log, sort_dicts=False)

assert review_final["review"] == review_resume_value
assert review_final["final_text"] == review_resume_value["edited_text"]
assert len(draft_entry_log) == 1
assert len(review_entry_log) == 2

print("\n友好摘要：已完成一次人工修订；上游草稿节点未重跑，审核节点重跑。")


首次 invoke 完整原始返回值：
{'topic': 'LangGraph interrupt',
 'draft': '主题：LangGraph interrupt。草稿结论：中断允许工作流等待人工反馈。',
 '__interrupt__': [Interrupt(value={'kind': 'content_review',
                                    'instruction': '请审核并返回修订后的正文',
                                    'draft': '主题：LangGraph '
                                             'interrupt。草稿结论：中断允许工作流等待人工反馈。'},
                             id='2f4eb6128e9720dc37c6f427f44e4413')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'content_review',
                  'instruction': '请审核并返回修订后的正文',
                  'draft': '主题：LangGraph interrupt。草稿结论：中断允许工作流等待人工反馈。'},
           id='2f4eb6128e9720dc37c6f427f44e4413')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={'topic': 'LangGraph interrupt', 'draft': '主题：LangGraph interrupt。草稿结论：中断允许工作流等待人工反馈。'}, next=('review_and_edit',), config={'configurable': {'thread_id': 'interrupt-review-edit-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-10aa-612e-8001-7a05e7b61653'}}, metadat

### 结果解读

暂停结果已经包含上游生成的完整 `draft`，而 `final_text` 要等人工恢复后才出现。恢复只重新调度尚未完成的审核节点，不会从图的 `START` 重新执行所有已完成节点，因此草稿节点入口 1 次、审核节点入口 2 次。


## 7. 案例 5：工具执行审批——副作用前确认

场景：流程准备调用名为 `safe_add` 的工具。人工先看到工具名和完整参数，批准后才进入独立工具节点。本例工具只是本地加法，没有网络或真实外部副作用。


In [6]:
class ToolApprovalState(TypedDict, total=False):
    tool_name: str
    arguments: dict[str, int]
    approval: dict[str, object]
    tool_result: dict[str, object]


tool_review_entry_log: list[str] = []
tool_execution_log: list[dict[str, object]] = []


def review_tool_call(state: ToolApprovalState) -> dict[str, object]:
    tool_review_entry_log.append(state["tool_name"])
    approval = interrupt(
        {
            "kind": "tool_approval",
            "tool_name": state["tool_name"],
            "arguments": state["arguments"],
            "question": "是否允许执行这个工具调用？",
        }
    )
    return {"approval": approval}


def route_tool_call(
    state: ToolApprovalState,
) -> Literal["execute_local_tool", "skip_local_tool"]:
    return (
        "execute_local_tool"
        if state["approval"]["approved"]
        else "skip_local_tool"
    )


def execute_local_tool(state: ToolApprovalState) -> dict[str, object]:
    arguments = state["arguments"]
    result = arguments["left"] + arguments["right"]
    record = {
        "tool_name": state["tool_name"],
        "arguments": arguments,
        "result": result,
    }
    tool_execution_log.append(record)
    return {"tool_result": record}


def skip_local_tool(_: ToolApprovalState) -> dict[str, object]:
    return {
        "tool_result": {
            "status": "skipped",
            "reason": "人工拒绝",
        }
    }


tool_builder = StateGraph(ToolApprovalState)
tool_builder.add_node("review_tool_call", review_tool_call)
tool_builder.add_node("execute_local_tool", execute_local_tool)
tool_builder.add_node("skip_local_tool", skip_local_tool)
tool_builder.add_edge(START, "review_tool_call")
tool_builder.add_conditional_edges(
    "review_tool_call",
    route_tool_call,
    {
        "execute_local_tool": "execute_local_tool",
        "skip_local_tool": "skip_local_tool",
    },
)
tool_builder.add_edge("execute_local_tool", END)
tool_builder.add_edge("skip_local_tool", END)
tool_saver = InMemorySaver()
tool_graph = tool_builder.compile(checkpointer=tool_saver)
tool_config = {
    "configurable": {"thread_id": "interrupt-tool-approval-v1"}
}

tool_first = tool_graph.invoke(
    {
        "tool_name": "safe_add",
        "arguments": {"left": 7, "right": 5},
    },
    tool_config,
    durability="sync",
)
tool_paused = tool_graph.get_state(tool_config)
print("首次 invoke 完整原始返回值：")
pprint(tool_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(tool_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(tool_paused, sort_dicts=False)

tool_resume_value = {
    "approved": True,
    "reviewer": "工具管理员",
    "reason": "参数只执行本地加法",
}
tool_command = Command(resume=tool_resume_value)
print("\n恢复使用的完整 Command：")
pprint(tool_command, sort_dicts=False)

tool_final = tool_graph.invoke(
    tool_command,
    tool_config,
    durability="sync",
)
tool_final_snapshot = tool_graph.get_state(tool_config)
print("\n恢复后的完整最终 State：")
pprint(tool_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(tool_final_snapshot, sort_dicts=False)
print("\n完整审批节点入口日志：")
pprint(tool_review_entry_log, sort_dicts=False)
print("\n完整工具执行日志：")
pprint(tool_execution_log, sort_dicts=False)

assert tool_final["approval"] == tool_resume_value
assert tool_final["tool_result"]["result"] == 12
assert len(tool_review_entry_log) == 2
assert len(tool_execution_log) == 1

print("\n友好摘要：工具经人工批准后执行；审核节点 2 次，工具节点 1 次。")


首次 invoke 完整原始返回值：
{'tool_name': 'safe_add',
 'arguments': {'left': 7, 'right': 5},
 '__interrupt__': [Interrupt(value={'kind': 'tool_approval',
                                    'tool_name': 'safe_add',
                                    'arguments': {'left': 7, 'right': 5},
                                    'question': '是否允许执行这个工具调用？'},
                             id='c08ab6134d8221fba93c7958459841ca')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'tool_approval',
                  'tool_name': 'safe_add',
                  'arguments': {'left': 7, 'right': 5},
                  'question': '是否允许执行这个工具调用？'},
           id='c08ab6134d8221fba93c7958459841ca')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={'tool_name': 'safe_add', 'arguments': {'left': 7, 'right': 5}}, next=('review_tool_call',), config={'configurable': {'thread_id': 'interrupt-tool-approval-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-10cc-6f9e-8000-91d122d6c49c'}}, metadata={'source': 'loop', 'step': 

### 结果解读

首次暂停时工具执行日志为空，证明未获得人工许可前没有调用工具。恢复后条件路由进入独立执行节点，结果为本地确定值 `12`。真实工具可能写数据库、发请求或付款；即使放到独立节点，也应以 `operation_id`、tool call ID 或业务唯一键实现幂等。


## 8. 案例 6：单节点串行中断——保持调用次序稳定

同一节点依次收集姓名、年龄、角色。每次恢复都会从函数开头重跑；运行时按 `interrupt()` 的调用索引重放已收到的历史恢复值，直到遇到下一次尚无答案的中断。


In [7]:
class SerialState(TypedDict, total=False):
    name: str
    age: int
    role: str


serial_entry_log: list[int] = []


def collect_profile_serially(_: SerialState) -> dict[str, object]:
    serial_entry_log.append(len(serial_entry_log) + 1)
    name = interrupt(
        {"step": 1, "field": "name", "question": "请输入姓名"}
    )
    age = interrupt(
        {"step": 2, "field": "age", "question": "请输入年龄"}
    )
    role = interrupt(
        {"step": 3, "field": "role", "question": "请输入角色"}
    )
    return {"name": name, "age": age, "role": role}


serial_builder = StateGraph(SerialState)
serial_builder.add_node(
    "collect_profile_serially", collect_profile_serially
)
serial_builder.add_edge(START, "collect_profile_serially")
serial_builder.add_edge("collect_profile_serially", END)
serial_saver = InMemorySaver()
serial_graph = serial_builder.compile(checkpointer=serial_saver)
serial_config = {
    "configurable": {"thread_id": "interrupt-serial-input-v1"}
}

serial_first = serial_graph.invoke(
    {}, serial_config, durability="sync"
)
serial_pause_1 = serial_graph.get_state(serial_config)
print("第一次中断的完整原始返回值：")
pprint(serial_first, sort_dicts=False)
print("\n第一次暂停的完整 StateSnapshot：")
pprint(serial_pause_1, sort_dicts=False)

serial_command_1 = Command(resume="Lin")
print("\n第一次恢复使用的完整 Command：")
pprint(serial_command_1, sort_dicts=False)
serial_second = serial_graph.invoke(
    serial_command_1, serial_config, durability="sync"
)
serial_pause_2 = serial_graph.get_state(serial_config)
print("\n第二次中断的完整原始返回值：")
pprint(serial_second, sort_dicts=False)
print("\n第二次暂停的完整 StateSnapshot：")
pprint(serial_pause_2, sort_dicts=False)

serial_command_2 = Command(resume=28)
print("\n第二次恢复使用的完整 Command：")
pprint(serial_command_2, sort_dicts=False)
serial_third = serial_graph.invoke(
    serial_command_2, serial_config, durability="sync"
)
serial_pause_3 = serial_graph.get_state(serial_config)
print("\n第三次中断的完整原始返回值：")
pprint(serial_third, sort_dicts=False)
print("\n第三次暂停的完整 StateSnapshot：")
pprint(serial_pause_3, sort_dicts=False)

serial_command_3 = Command(resume="editor")
print("\n第三次恢复使用的完整 Command：")
pprint(serial_command_3, sort_dicts=False)
serial_final = serial_graph.invoke(
    serial_command_3, serial_config, durability="sync"
)
serial_final_snapshot = serial_graph.get_state(serial_config)
print("\n恢复后的完整最终 State：")
pprint(serial_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(serial_final_snapshot, sort_dicts=False)
print("\n完整节点入口日志：")
pprint(serial_entry_log, sort_dicts=False)

assert serial_first["__interrupt__"][0].value["field"] == "name"
assert serial_second["__interrupt__"][0].value["field"] == "age"
assert serial_third["__interrupt__"][0].value["field"] == "role"
assert serial_final == {"name": "Lin", "age": 28, "role": "editor"}
assert len(serial_entry_log) == 4

print("\n友好摘要：三次恢复依次填入三个 interrupt；节点共从开头进入 4 次。")


第一次中断的完整原始返回值：
{'__interrupt__': [Interrupt(value={'step': 1,
                                    'field': 'name',
                                    'question': '请输入姓名'},
                             id='e99b4f6e598e92571bbfd3dbcd78f79a')]}

第一次暂停的完整 StateSnapshot：
StateSnapshot(values={}, next=('collect_profile_serially',), config={'configurable': {'thread_id': 'interrupt-serial-input-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-10ea-6904-8000-947cb050546b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-21T09:46:59.498004+00:00', parent_config={'configurable': {'thread_id': 'interrupt-serial-input-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-10e8-6b36-bfff-012f6dcc9d2d'}}, tasks=(PregelTask(id='194e36dd-3f71-53d0-b616-01c83a7179a4', name='collect_profile_serially', path=('__pregel_pull', 'collect_profile_serially'), error=None, interrupts=(Interrupt(value={'step': 1, 'field': 'name', 'question': '请输入姓名'}, id='e99b4f6e598e92571bbfd3dbcd7

### 结果解读

第一次恢复时，历史值 `"Lin"` 满足第一个 `interrupt()`，随后在第二个中断处再次返回调用者；后续同理。这里不能根据状态、随机数、当前时间或可变列表，在恢复前动态插入、跳过或重排中断，否则历史 resume 值会匹配到错误的位置。


## 9. 案例 7：人工输入校验——非法值再次中断

调用方第一次提供评分 `9`，节点判定不在 1～5 之间，于是带着错误信息再次中断；第二次提供 `4` 后正常完成。校验循环内的中断顺序是固定的：每轮只追加下一次调用，不会根据恢复期间变化的外部集合重排历史调用。


In [8]:
class ValidationState(TypedDict, total=False):
    rating: int


validation_entry_log: list[int] = []


def collect_valid_rating(_: ValidationState) -> dict[str, int]:
    validation_entry_log.append(len(validation_entry_log) + 1)
    attempt = 1
    error: str | None = None
    while True:
        value = interrupt(
            {
                "kind": "rating",
                "attempt": attempt,
                "question": "请给出 1～5 的整数评分",
                "error": error,
            }
        )
        is_valid = (
            isinstance(value, int)
            and not isinstance(value, bool)
            and 1 <= value <= 5
        )
        if is_valid:
            return {"rating": value}
        error = f"{value!r} 不是 1～5 的整数"
        attempt += 1


validation_builder = StateGraph(ValidationState)
validation_builder.add_node(
    "collect_valid_rating", collect_valid_rating
)
validation_builder.add_edge(START, "collect_valid_rating")
validation_builder.add_edge("collect_valid_rating", END)
validation_saver = InMemorySaver()
validation_graph = validation_builder.compile(
    checkpointer=validation_saver
)
validation_config = {
    "configurable": {"thread_id": "interrupt-validation-v1"}
}

validation_first = validation_graph.invoke(
    {}, validation_config, durability="sync"
)
validation_pause_1 = validation_graph.get_state(validation_config)
print("第一次中断的完整原始返回值：")
pprint(validation_first, sort_dicts=False)
print("\n第一次暂停的完整 StateSnapshot：")
pprint(validation_pause_1, sort_dicts=False)

invalid_command = Command(resume=9)
print("\n无效输入使用的完整 Command：")
pprint(invalid_command, sort_dicts=False)
validation_second = validation_graph.invoke(
    invalid_command,
    validation_config,
    durability="sync",
)
validation_pause_2 = validation_graph.get_state(validation_config)
print("\n校验失败后再次中断的完整原始返回值：")
pprint(validation_second, sort_dicts=False)
print("\n第二次暂停的完整 StateSnapshot：")
pprint(validation_pause_2, sort_dicts=False)

valid_command = Command(resume=4)
print("\n有效输入使用的完整 Command：")
pprint(valid_command, sort_dicts=False)
validation_final = validation_graph.invoke(
    valid_command,
    validation_config,
    durability="sync",
)
validation_final_snapshot = validation_graph.get_state(
    validation_config
)
print("\n恢复后的完整最终 State：")
pprint(validation_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(validation_final_snapshot, sort_dicts=False)
print("\n完整节点入口日志：")
pprint(validation_entry_log, sort_dicts=False)

second_prompt = validation_second["__interrupt__"][0].value
assert second_prompt["attempt"] == 2
assert "9" in second_prompt["error"]
assert validation_final == {"rating": 4}
assert len(validation_entry_log) == 3

print("\n友好摘要：无效值触发第二次中断，有效值 4 最终写入 State。")


第一次中断的完整原始返回值：
{'__interrupt__': [Interrupt(value={'kind': 'rating',
                                    'attempt': 1,
                                    'question': '请给出 1～5 的整数评分',
                                    'error': None},
                             id='6e3752b548abb8621e93925febce3462')]}

第一次暂停的完整 StateSnapshot：
StateSnapshot(values={}, next=('collect_valid_rating',), config={'configurable': {'thread_id': 'interrupt-validation-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1109-6f8e-8000-72ce5374f2b3'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-21T09:46:59.510871+00:00', parent_config={'configurable': {'thread_id': 'interrupt-validation-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1108-6b20-bfff-40e8bed26ae3'}}, tasks=(PregelTask(id='e20fdaa7-dbce-ff4a-d4e5-a64684522b01', name='collect_valid_rating', path=('__pregel_pull', 'collect_valid_rating'), error=None, interrupts=(Interrupt(value={'kind': 'rating', 'attempt': 1, 'q

### 结果解读

第二次 `invoke()` 仍返回 `__interrupt__`，而不是错误终止；新的 `Interrupt.value` 带有 `attempt=2` 和完整错误消息。第三次恢复时，节点再次从开头执行，先重放历史无效值，再在第二个 `interrupt()` 取得新值 `4`。有些版本/时点下，带 pending write 的最新 `get_state()` 对 `next` 的呈现可能与首次暂停不同，因此判断是否仍中断应直接查看返回值及 `snapshot.tasks[*].interrupts`，不要只靠一个摘要字段。


## 10. Checkpoint 专题：同一超步只有部分任务中断

两个并行任务位于同一超步：`load_policy` 正常完成，`review_draft` 触发中断。运行时会等待并收集该超步的任务结果；正常任务的结果与中断任务的元数据一起进入 checkpoint。恢复时，已成功任务的 pending write 会被复用，不需要再次运行。


In [9]:
class PartialStepState(TypedDict, total=False):
    draft: str
    policy: str
    reviewed_draft: str


policy_entry_log: list[str] = []
partial_review_entry_log: list[str] = []


def load_policy(_: PartialStepState) -> dict[str, str]:
    policy_entry_log.append("load_policy")
    return {"policy": "local-policy-v1"}


def review_draft(state: PartialStepState) -> dict[str, object]:
    partial_review_entry_log.append("review_draft")
    reviewed = interrupt(
        {
            "kind": "draft_review",
            "draft": state["draft"],
            "question": "请返回复核后的草稿",
        }
    )
    return {"reviewed_draft": reviewed}


partial_builder = StateGraph(PartialStepState)
partial_builder.add_node("load_policy", load_policy)
partial_builder.add_node("review_draft", review_draft)
partial_builder.add_edge(START, "load_policy")
partial_builder.add_edge(START, "review_draft")
partial_builder.add_edge("load_policy", END)
partial_builder.add_edge("review_draft", END)
partial_saver = InMemorySaver()
partial_graph = partial_builder.compile(checkpointer=partial_saver)
partial_config = {
    "configurable": {
        "thread_id": "interrupt-partial-superstep-v1"
    }
}

partial_first = partial_graph.invoke(
    {"draft": "原始草稿"},
    partial_config,
    durability="sync",
)
partial_paused = partial_graph.get_state(partial_config)
print("首次 invoke 完整原始返回值：")
pprint(partial_first, sort_dicts=False)
print("\n完整 __interrupt__ 字段：")
pprint(partial_first["__interrupt__"], sort_dicts=False)
print("\n暂停后的完整 StateSnapshot：")
pprint(partial_paused, sort_dicts=False)
print("\n暂停快照的完整 tasks 字段：")
pprint(partial_paused.tasks, sort_dicts=False)

partial_command = Command(resume="人工复核后的草稿")
print("\n恢复使用的完整 Command：")
pprint(partial_command, sort_dicts=False)
partial_final = partial_graph.invoke(
    partial_command,
    partial_config,
    durability="sync",
)
partial_final_snapshot = partial_graph.get_state(partial_config)
print("\n恢复后的完整最终 State：")
pprint(partial_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(partial_final_snapshot, sort_dicts=False)
print("\n完整正常分支入口日志：")
pprint(policy_entry_log, sort_dicts=False)
print("\n完整中断分支入口日志：")
pprint(partial_review_entry_log, sort_dicts=False)

task_by_name = {task.name: task for task in partial_paused.tasks}
assert task_by_name["load_policy"].result == {
    "policy": "local-policy-v1"
}
assert task_by_name["review_draft"].interrupts
assert partial_final["reviewed_draft"] == "人工复核后的草稿"
assert len(policy_entry_log) == 1
assert len(partial_review_entry_log) == 2

print("\n友好摘要：正常并行任务只执行 1 次，中断任务在恢复时从头重跑。")


首次 invoke 完整原始返回值：
{'draft': '原始草稿',
 'policy': 'local-policy-v1',
 '__interrupt__': [Interrupt(value={'kind': 'draft_review',
                                    'draft': '原始草稿',
                                    'question': '请返回复核后的草稿'},
                             id='2be1b73d3ad430840943acbc95f49e9e')]}

完整 __interrupt__ 字段：
[Interrupt(value={'kind': 'draft_review',
                  'draft': '原始草稿',
                  'question': '请返回复核后的草稿'},
           id='2be1b73d3ad430840943acbc95f49e9e')]

暂停后的完整 StateSnapshot：
StateSnapshot(values={'draft': '原始草稿', 'policy': 'local-policy-v1'}, next=('review_draft',), config={'configurable': {'thread_id': 'interrupt-partial-superstep-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1127-6b42-8000-501ba5c345dd'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-21T09:46:59.523048+00:00', parent_config={'configurable': {'thread_id': 'interrupt-partial-superstep-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d4

### 结果解读

首次 `invoke()` 已返回 `policy`，同时带有 `__interrupt__`。暂停快照中，`load_policy` 任务的 `result` 是完整字典，而 `review_draft` 任务的 `interrupts` 保存中断对象。恢复后 `load_policy` 入口日志仍只有一项，证明同一超步内已成功的并行任务不会仅因另一任务中断而被无条件重跑。


## 11. 使用规范与反例

### 11.1 不要在节点内用宽泛 `try/except` 包住 `interrupt()`

暂停依赖内部控制流异常传播。下面是**不要执行的反例**：

```python
def bad_node(state):
    try:
        answer = interrupt({"question": "是否继续？"})
    except Exception:  # 会吞掉 LangGraph 用来通知运行时的内部异常
        answer = "fallback"
    return {"answer": answer}
```

可以在 `interrupt()` 已经返回之后，对真正可能失败的普通业务调用做窄范围异常处理；也可以在图外调用边界捕获你明确预期的异常类型。这和包裹节点内的 `interrupt()` 不是一回事。

### 11.2 不要在恢复前改变调用顺序或数量

串行案例使用固定的 `name -> age -> role` 次序。下面的模式不安全：

```python
# 反例：items 或条件在暂停期间变化，会让历史 resume 值错配。
for item in load_mutable_items_from_outside():
    answers[item] = interrupt({"item": item})
```

若问题集合本身会变化，应把每个问题建模为独立任务/节点，或在开始前把稳定问题列表写入 State，并保证恢复期间不改变该调用序列。

### 11.3 中断载荷保持可序列化

本章所有 `interrupt(value)` 都只传字符串、数字、布尔值、列表或由这些值组成的字典。不要把打开的文件、函数、数据库连接或任意自定义实例作为中断载荷。序列化约束既影响调用者能否收到值，也影响持久化 checkpointer 的兼容性。

### 11.4 `interrupt()` 前的副作用必须幂等

基础、审批、工具和串行案例的入口日志已直接证明：中断节点会从开头重跑。工程上可采用：

- 用稳定 `operation_id` 做幂等键，并在外部系统建立唯一约束；
- 把非幂等动作移到中断后的独立节点；
- 使用幂等 API、去重表或 transactional outbox；
- 仍按 retry、replay、time travel 和 worker 崩溃会重复执行来设计，不能把独立节点误当成 exactly-once 保证。


## 12. 边界验证：没有 checkpointer 只能“看见中断”，不能恢复

LangGraph 1.1.2 中，不带 checkpointer 的图第一次仍可能返回 `__interrupt__`；但没有可恢复线程时间线，随后传入 `Command(resume=...)` 会失败。下面的异常捕获位于**图外调用边界**，并且只捕获明确预期的 `RuntimeError`，没有包裹节点里的 `interrupt()`。


In [10]:
class NoCheckpointState(TypedDict, total=False):
    prompt: str
    answer: str


def pause_without_checkpoint(
    state: NoCheckpointState,
) -> dict[str, object]:
    answer = interrupt(state["prompt"])
    return {"answer": answer}


no_checkpoint_builder = StateGraph(NoCheckpointState)
no_checkpoint_builder.add_node(
    "pause_without_checkpoint", pause_without_checkpoint
)
no_checkpoint_builder.add_edge(START, "pause_without_checkpoint")
no_checkpoint_builder.add_edge("pause_without_checkpoint", END)
no_checkpoint_graph = no_checkpoint_builder.compile()
no_checkpoint_config = {
    "configurable": {"thread_id": "interrupt-no-checkpointer-v1"}
}

no_checkpoint_first = no_checkpoint_graph.invoke(
    {"prompt": "请输入任意回答"}, no_checkpoint_config
)
print("没有 checkpointer 时首次调用的完整返回值：")
pprint(no_checkpoint_first, sort_dicts=False)

try:
    no_checkpoint_graph.invoke(
        Command(resume="无法恢复的回答"),
        no_checkpoint_config,
    )
except RuntimeError as exc:
    resume_without_checkpoint_error = exc
else:
    resume_without_checkpoint_error = None

print("\n恢复失败的完整异常对象：")
pprint(resume_without_checkpoint_error)
assert "__interrupt__" in no_checkpoint_first
assert isinstance(resume_without_checkpoint_error, RuntimeError)

print("\n友好摘要：首次可见中断，但恢复失败；异常类型为 RuntimeError。")


没有 checkpointer 时首次调用的完整返回值：
{'prompt': '请输入任意回答',
 '__interrupt__': [Interrupt(value='请输入任意回答',
                             id='28565f570a4df3bf0b1e3f8a0bb3ea49')]}

恢复失败的完整异常对象：
RuntimeError('Cannot use Command(resume=...) without checkpointer')

友好摘要：首次可见中断，但恢复失败；异常类型为 RuntimeError。


## 13. 静态断点对照：调试暂停不是动态 HITL

静态断点可在 `compile()` 或单次 `invoke()` 时配置，并可用 `interrupt_before` / `interrupt_after` 观察超步边界。它同样需要 checkpointer，但不会向调用者暴露问题，也不接收人工 resume 值；恢复输入是 `None`，而不是 `Command(resume=...)`。

页面还演示了顺序/并行图、编译期/调用期配置，以及调用期恢复时必须保持断点设置的一致性。这些都是调试策略，不应与业务审批、编辑或输入收集混为一谈。下面只用最小顺序图验证核心差异。


In [11]:
class StaticState(TypedDict, total=False):
    phase: str


static_entry_log: list[str] = []


def static_prepare(_: StaticState) -> dict[str, str]:
    static_entry_log.append("static_prepare")
    return {"phase": "prepared"}


def static_finalize(_: StaticState) -> dict[str, str]:
    static_entry_log.append("static_finalize")
    return {"phase": "finished"}


static_builder = StateGraph(StaticState)
static_builder.add_node("static_prepare", static_prepare)
static_builder.add_node("static_finalize", static_finalize)
static_builder.add_edge(START, "static_prepare")
static_builder.add_edge("static_prepare", "static_finalize")
static_builder.add_edge("static_finalize", END)
static_saver = InMemorySaver()
static_graph = static_builder.compile(
    checkpointer=static_saver,
    interrupt_before=["static_finalize"],
)
static_config = {
    "configurable": {"thread_id": "static-breakpoint-v1"}
}

static_first = static_graph.invoke(
    {}, static_config, durability="sync"
)
static_paused = static_graph.get_state(static_config)
print("静态断点首次 invoke 的完整原始返回值：")
pprint(static_first, sort_dicts=False)
print("\n静态暂停后的完整 StateSnapshot：")
pprint(static_paused, sort_dicts=False)

static_final = static_graph.invoke(
    None, static_config, durability="sync"
)
static_final_snapshot = static_graph.get_state(static_config)
print("\n以 None 恢复后的完整最终 State：")
pprint(static_final, sort_dicts=False)
print("\n恢复后的完整 StateSnapshot：")
pprint(static_final_snapshot, sort_dicts=False)
print("\n完整静态节点入口日志：")
pprint(static_entry_log, sort_dicts=False)

assert "__interrupt__" not in static_first
assert static_paused.next == ("static_finalize",)
assert static_final == {"phase": "finished"}
assert static_entry_log == ["static_prepare", "static_finalize"]

print("\n友好摘要：静态断点没有反馈载荷，使用 None 从超步边界恢复。")


静态断点首次 invoke 的完整原始返回值：
{'phase': 'prepared'}

静态暂停后的完整 StateSnapshot：
StateSnapshot(values={'phase': 'prepared'}, next=('static_finalize',), config={'configurable': {'thread_id': 'static-breakpoint-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1155-6c68-8001-be7be53531a2'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-21T09:46:59.541920+00:00', parent_config={'configurable': {'thread_id': 'static-breakpoint-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1154-6dd6-8000-9fce80ac87bb'}}, tasks=(PregelTask(id='0e490d53-f198-475f-c08c-336cb543935f', name='static_finalize', path=('__pregel_pull', 'static_finalize'), error=None, interrupts=(), state=None, result=None),), interrupts=())

以 None 恢复后的完整最终 State：
{'phase': 'finished'}

恢复后的完整 StateSnapshot：
StateSnapshot(values={'phase': 'finished'}, next=(), config={'configurable': {'thread_id': 'static-breakpoint-v1', 'checkpoint_ns': '', 'checkpoint_id': '1f19d454-1158-6c10-8002-f358e4e99400'}}, met

## 14. `InMemorySaver` 的生命周期与适用边界

本章每个动态案例都使用独立 `InMemorySaver` 和独立 `thread_id`，避免不同工作流的 checkpoint 相互污染。恢复同时依赖：

- 相同 `thread_id`；
- 仍持有原 checkpoint 的同一个 saver 实例；
- 与暂停时兼容的图结构和中断调用顺序。

`InMemorySaver` 适合 Notebook、单元测试和单进程调试。内核重启、进程退出、新建 saver 或切换到另一进程后，原内存 checkpoint 不再可用。长时间人工审批应使用持久化 checkpointer，并另行设计并发控制、权限、审计、超时和数据保留策略。


## 15. 最终功能核对

前面已先后输出每个案例的完整原始返回值、中断列表、暂停快照、恢复命令和最终状态。下面汇总关键断言；摘要不能替代上面的原始证据。


In [12]:
verification = {
    "案例1_基础恢复": basic_final["human_note"] == basic_resume_value,
    "案例2_并行ID映射": parallel_final == parallel_answers,
    "案例3_审批路由": approval_final["outcome"].startswith("已批准"),
    "案例4_审核编辑": review_final["final_text"]
    == review_resume_value["edited_text"],
    "案例5_工具审批": tool_final["tool_result"]["result"] == 12,
    "案例6_串行三次中断": serial_final
    == {"name": "Lin", "age": 28, "role": "editor"},
    "案例7_输入校验重中断": validation_final == {"rating": 4},
    "部分超步_成功任务未重跑": len(policy_entry_log) == 1,
    "无checkpoint_恢复失败": isinstance(
        resume_without_checkpoint_error, RuntimeError
    ),
    "静态断点_None恢复": static_final == {"phase": "finished"},
}

print("完整核对字典：")
pprint(verification, sort_dicts=False)
assert all(verification.values())
print("\n友好摘要：全部关键断言通过。")


完整核对字典：
{'案例1_基础恢复': True,
 '案例2_并行ID映射': True,
 '案例3_审批路由': True,
 '案例4_审核编辑': True,
 '案例5_工具审批': True,
 '案例6_串行三次中断': True,
 '案例7_输入校验重中断': True,
 '部分超步_成功任务未重跑': True,
 '无checkpoint_恢复失败': True,
 '静态断点_None恢复': True}

友好摘要：全部关键断言通过。


## 16. 总结

- 动态 `interrupt()` 是业务逻辑中的暂停点；首次 `invoke()` 返回 `__interrupt__`，不会一直占住 Python 调用；
- checkpointer 保存 State、任务与中断元数据，不保存 Python 栈；恢复时中断节点从函数开头重跑；
- 必须复用相同 `thread_id`，并用 `Command(resume=value)` 恢复；到同一次 `interrupt()` 时，`value` 成为返回值；
- 并行中断用 `Interrupt.id -> value` 映射；串行中断必须维持稳定调用顺序；校验失败可以再次中断；
- 审批、编辑和工具执行前确认只是同一机制在不同业务边界上的组合；
- `interrupt()` 前副作用必然面临重复执行，独立节点只能减少一种重复来源，不能替代幂等设计；
- 静态断点服务于调试，使用 `None` 恢复，不是接收人类反馈的动态 HITL。

### 延伸资料

- [LangGraph：Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangGraph API Reference：interrupt](https://reference.langchain.com/python/langgraph/types/interrupt)
- [LangGraph API Reference：Command](https://reference.langchain.com/python/langgraph/types/Command)
- [LangGraph：Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
